# CropCop Track B — 00 Readiness & Immutable Materialization

**Purpose:** construct the two immutable, content-addressed Track-B v5 Kaggle input bundles. This notebook performs **zero protected external R07 predictions**.

### Kaggle setup
- Account: `ranamuhammadahmed6`
- Accelerator: **T4 x2**
- Internet: **ON**
- Secret configured: **KAGGLE_API_TOKEN**
- Attach exactly these five source datasets:
  1. `ranamuhammadahmed6/cropcop-finalized-v8-11-2026-1`
  2. `sabahatabbas/sec-je-r07-cnxtt-context-s1-8904b100d223-a01`
  3. `sabahatabbas/cropcop-r07-cnxtt-context-s2-abce1197-56023042`
  4. `sabahatabbas/cropcop-r07-cnxtt-context-s3-f13ca687-56023042`
  5. `ranamuhammadahmed6/cropcop-secondary-g1-8904b100` (**Version 2**)

Do not attach GVLiD or Irish Potato manually. The notebook checks out one exact detached runtime commit, verifies its v5 lock/attestation chain before runtime repair, acquires and seals the published external cohorts before expensive historical computation, publishes content-addressed private bundles, and round-trip verifies them.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys

WORK = Path('/kaggle/working')
REPO = WORK / 'ResearchWork-CropCop-trackb-v5-runtime'
REPO_URL = 'https://github.com/rana-m-ahmed/ResearchWork-CropCop.git'
SOURCE_COMMIT = 'eacae1d5b3a9012d65d4be3152e93cf03eefceb7'
RELEASE_ID = 'TRACKB_V5_RELEASE_AUTHORITY_v1'

if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git','clone','--filter=blob:none','--no-checkout',REPO_URL,str(REPO)], check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',SOURCE_COMMIT], check=True)
head = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip()
if head != SOURCE_COMMIT:
    raise RuntimeError(f'Frozen v5 runtime source mismatch: expected {SOURCE_COMMIT}, got {head}')

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

LOCK = REPO / 'journal_extension/track_b_r07/TRACKB_R07_EXECUTION_LOCK_v4.json'
ATTEST = REPO / 'journal_extension/track_b_r07/TRACKB_CODE_ATTESTATION_v4.json'
SOURCE_LOCK = REPO / 'journal_extension/track_b_r07/TRACKB_EXTERNAL_SOURCE_LOCK_v2.json'
MATERIALIZATION_LOCK = REPO / 'journal_extension/track_b_r07/TRACKB_INPUT_MATERIALIZATION_LOCK_v2.json'
for path in (LOCK, ATTEST, SOURCE_LOCK, MATERIALIZATION_LOCK):
    if not path.is_file():
        raise RuntimeError(f'Frozen v5 runtime authority missing: {path}')

execution_lock = json.loads(LOCK.read_text(encoding='utf-8'))
attestation = json.loads(ATTEST.read_text(encoding='utf-8'))
source_lock = json.loads(SOURCE_LOCK.read_text(encoding='utf-8'))
materialization_lock = json.loads(MATERIALIZATION_LOCK.read_text(encoding='utf-8'))

if execution_lock.get('lock_id') != 'TRACKB_R07_EXECUTION_LOCK_v4':
    raise RuntimeError('Wrong Track-B execution lock in frozen runtime')
if attestation.get('attestation_id') != 'TRACKB_CODE_ATTESTATION_v4':
    raise RuntimeError('Wrong Track-B code attestation in frozen runtime')
if source_lock.get('status') != 'PASS_EXTERNAL_SOURCE_AUTHORITY':
    raise RuntimeError('External source authority is not PASS')
if materialization_lock.get('lock_id') != 'TRACKB_INPUT_MATERIALIZATION_LOCK_v2':
    raise RuntimeError('Wrong Track-B materialization lock in frozen runtime')
if sha256_file(ATTEST) != execution_lock.get('code_attestation_sha256'):
    raise RuntimeError('Execution lock does not bind the frozen runtime code attestation')

for row in attestation.get('files') or []:
    rel = str(row.get('path', '')).strip()
    expected = str(row.get('git_blob_sha1', '')).lower()
    path = (REPO / rel).resolve()
    if REPO not in path.parents or not path.is_file():
        raise RuntimeError(f'Attested runtime file missing/unsafe: {rel}')
    observed = subprocess.check_output(
        ['git','-C',str(REPO),'hash-object',rel], text=True
    ).strip().lower()
    if observed != expected:
        raise RuntimeError(f'Frozen runtime attestation mismatch: {rel}')

from kaggle_secrets import UserSecretsClient
token = (UserSecretsClient().get_secret('KAGGLE_API_TOKEN') or '').strip()
if not token or any(ch.isspace() for ch in token):
    raise RuntimeError('KAGGLE_API_TOKEN is missing/invalid')
os.environ['KAGGLE_API_TOKEN'] = token
del token

print(json.dumps({
    'status': 'PASS_FROZEN_V5_RUNTIME_AUTHORITY',
    'release_id': RELEASE_ID,
    'runtime_source_commit': head,
    'execution_lock': execution_lock['lock_id'],
    'code_attestation': attestation['attestation_id'],
    'external_source_authority': source_lock['status'],
    'materialization_lock': materialization_lock['lock_id'],
}, indent=2, sort_keys=True))


In [ ]:
BOOTSTRAP = REPO / 'journal_extension/scripts/bootstrap_trackb_runtime.py'
LOCKFILE = REPO / 'journal_extension/track_b_r07/requirements-trackb.lock.txt'
RECEIPT = WORK / 'TRACKB_V5_READINESS_RUNTIME.json'

subprocess.run([
    sys.executable, str(BOOTSTRAP),
    '--requirements', str(LOCKFILE),
    '--receipt', str(RECEIPT),
], cwd=REPO, check=True, env=os.environ.copy())

runtime = json.loads(RECEIPT.read_text(encoding='utf-8'))
if runtime.get('status') != 'PASS':
    raise RuntimeError('Track-B v5 runtime bootstrap failed')
if runtime.get('scientific_execution_requires_fresh_subprocess') is not True:
    raise RuntimeError('Runtime bootstrap does not require a fresh scientific subprocess')
print(json.dumps({
    'runtime_status': runtime['status'],
    'cuda_available': runtime['probe']['cuda_available'],
    'cuda_devices': runtime['probe']['cuda_devices'],
    'versions': runtime['probe']['versions'],
}, indent=2, sort_keys=True))


In [ ]:
MATERIALIZER = REPO / 'journal_extension/scripts/trackb_v4_materialize.py'
OUT = Path('/kaggle/tmp/trackb_v5_materialization')
if OUT.exists():
    shutil.rmtree(OUT)

subprocess.run([
    sys.executable, str(MATERIALIZER),
    '--repo-root', str(REPO),
    '--input-root', '/kaggle/input',
    '--output-root', str(OUT),
    '--device', 'cuda:0',
    '--kaggle-owner', 'AUTO',
], cwd=REPO, check=True, env=os.environ.copy())


In [ ]:
receipt_path = Path('/kaggle/tmp/trackb_v5_materialization') / 'TRACKB_READINESS_RECEIPT.json'
receipt = json.loads(receipt_path.read_text(encoding='utf-8'))
if receipt.get('status') != 'PASS_TRACKB_INPUT_MATERIALIZATION':
    raise RuntimeError(f"Readiness did not PASS: {receipt.get('status')}")
if receipt.get('protected_external_prediction_count') != 0:
    raise RuntimeError('Readiness produced protected external predictions')
if receipt.get('v1_test_accessed') is not False:
    raise RuntimeError('Readiness reports V1-test access')
if receipt['materialization']['repository_source_sha'] != SOURCE_COMMIT:
    raise RuntimeError('Readiness materialization used the wrong runtime source commit')

durable_receipt = Path('/kaggle/working/TRACKB_V5_READINESS_RECEIPT.json')
shutil.copy2(receipt_path, durable_receipt)

print(json.dumps({
    'status': receipt['status'],
    'release_id': RELEASE_ID,
    'materialization_id': receipt['materialization']['materialization_id'],
    'repository_source_sha': receipt['materialization']['repository_source_sha'],
    'source_qualification_sha256': receipt['source_qualification_sha256'],
    'external_source_lock_sha256': receipt['materialization']['external_source_lock_sha256'],
    'input_materialization_lock_sha256': receipt['materialization']['input_materialization_lock_sha256'],
    'infrastructure_dataset': receipt['publication']['infrastructure']['slug'],
    'external_dataset': receipt['publication']['external']['slug'],
    'protected_external_prediction_count': receipt['protected_external_prediction_count'],
    'v1_test_accessed': receipt['v1_test_accessed'],
    'durable_receipt': str(durable_receipt),
    'next_step': 'Attach these exact two content-addressed datasets to TrackB_01_Final_Execution.ipynb in qualification mode.',
}, indent=2, sort_keys=True))
